In [0]:
-- ============================================================
-- AEGIS SUPPLY CHAIN INTELLIGENCE PLATFORM
-- ============================================================
-- SILVER LAYER
--
-- Purpose:
--   Transform validated Bronze tables into clean,
--   analytics-ready entities and facts.
--
-- Silver responsibilities:
--   1. Standardize data types and text fields
--   2. Preserve the original business grain
--   3. Add deterministic row-level derived fields
--   4. Make downstream Gold analytics simpler and consistent
--
-- Silver does NOT:
--   - Aggregate business KPIs
--   - Perform AI/LLM reasoning
--   - Encode the injected incidents
--   - Use the ground-truth incident table
--
-- Ground truth incidents remain isolated for evaluation.
--
-- Architecture:
--
--   Bronze
--      ↓
--   Clean + Standardize + Derive
--      ↓
--   Silver
--      ↓
--   Aggregate Business Metrics
--      ↓
--   Gold
-- ============================================================


-- ============================================================
-- 1. DIMENSIONS
-- ============================================================

-- ------------------------------------------------------------
-- CUSTOMER
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.customer AS
SELECT
    CAST(customer_id AS STRING) AS customer_id,
    TRIM(region) AS region,
    TRIM(customer_segment) AS customer_segment,
    CAST(activity_level AS DOUBLE) AS activity_level
FROM workspace.aegis_bronze.customer;


-- ------------------------------------------------------------
-- PRODUCT
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.product AS
SELECT
    CAST(product_id AS STRING) AS product_id,
    TRIM(product_name) AS product_name,
    TRIM(category) AS category,
    CAST(unit_cost AS DOUBLE) AS unit_cost,
    CAST(selling_price AS DOUBLE) AS selling_price,
    CAST(demand_index AS DOUBLE) AS demand_index
FROM workspace.aegis_bronze.product;


-- ------------------------------------------------------------
-- SUPPLIER
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.supplier AS
SELECT
    CAST(supplier_id AS STRING) AS supplier_id,
    TRIM(supplier_name) AS supplier_name,
    TRIM(region) AS region,
    TRIM(supplier_type) AS supplier_type,
    CAST(reliability_score AS DOUBLE) AS reliability_score,
    CAST(lead_time_variability AS DOUBLE) AS lead_time_variability
FROM workspace.aegis_bronze.supplier;


-- ------------------------------------------------------------
-- SUPPLIER ↔ PRODUCT
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.supplier_product AS
SELECT
    CAST(supplier_id AS STRING) AS supplier_id,
    CAST(product_id AS STRING) AS product_id,
    CAST(agreed_lead_time AS BIGINT) AS agreed_lead_time,
    CAST(unit_cost AS DOUBLE) AS unit_cost,
    CAST(minimum_order_quantity AS BIGINT) AS minimum_order_quantity
FROM workspace.aegis_bronze.supplier_product;


-- ------------------------------------------------------------
-- WAREHOUSE
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.warehouse AS
SELECT
    CAST(warehouse_id AS STRING) AS warehouse_id,
    TRIM(region) AS region,
    CAST(capacity AS BIGINT) AS capacity,
    TRIM(warehouse_type) AS warehouse_type,
    CAST(processing_efficiency AS DOUBLE) AS processing_efficiency
FROM workspace.aegis_bronze.warehouse;


-- ------------------------------------------------------------
-- CARRIER
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.carrier AS
SELECT
    CAST(carrier_id AS STRING) AS carrier_id,
    TRIM(carrier_name) AS carrier_name,
    TRIM(service_type) AS service_type,
    CAST(base_transit_days AS DOUBLE) AS base_transit_days,
    CAST(transit_variability AS DOUBLE) AS transit_variability
FROM workspace.aegis_bronze.carrier;



-- ============================================================
-- 2. ORDER FACTS
-- ============================================================

-- ------------------------------------------------------------
-- ORDER
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.order AS
SELECT
    CAST(order_id AS STRING) AS order_id,
    CAST(customer_id AS STRING) AS customer_id,
    CAST(order_date AS TIMESTAMP) AS order_date,
    TRIM(region) AS region,
    TRIM(order_status) AS order_status
FROM workspace.aegis_bronze.order;


-- ------------------------------------------------------------
-- ORDER ITEM
--
-- Grain: one row per order item
--
-- revenue is calculated deterministically from the source
-- quantity and historical unit price.
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.order_item AS
SELECT
    CAST(order_item_id AS STRING) AS order_item_id,
    CAST(order_id AS STRING) AS order_id,
    CAST(product_id AS STRING) AS product_id,
    CAST(supplier_id AS STRING) AS supplier_id,
    CAST(quantity AS BIGINT) AS quantity,
    CAST(unit_price AS DOUBLE) AS unit_price,

    ROUND(
        CAST(quantity AS DOUBLE) * CAST(unit_price AS DOUBLE),
        2
    ) AS revenue

FROM workspace.aegis_bronze.order_item;



-- ============================================================
-- 3. SHIPMENT FACTS
-- ============================================================

-- ------------------------------------------------------------
-- SHIPMENT
--
-- Derived metrics:
--   expected_transit_days
--   actual_transit_days
--   delivery_delay_days
--   is_late
--
-- No aggregation happens here.
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.shipment AS
SELECT
    CAST(shipment_id AS STRING) AS shipment_id,
    CAST(order_id AS STRING) AS order_id,
    CAST(warehouse_id AS STRING) AS warehouse_id,
    CAST(carrier_id AS STRING) AS carrier_id,

    CAST(shipment_date AS TIMESTAMP) AS shipment_date,
    CAST(expected_delivery_date AS TIMESTAMP) AS expected_delivery_date,
    CAST(actual_delivery_date AS TIMESTAMP) AS actual_delivery_date,

    TRIM(delivery_status) AS delivery_status,

    DATEDIFF(
        CAST(expected_delivery_date AS DATE),
        CAST(shipment_date AS DATE)
    ) AS expected_transit_days,

    CASE
        WHEN actual_delivery_date IS NOT NULL
        THEN DATEDIFF(
            CAST(actual_delivery_date AS DATE),
            CAST(shipment_date AS DATE)
        )
    END AS actual_transit_days,

    CASE
        WHEN actual_delivery_date IS NOT NULL
        THEN DATEDIFF(
            CAST(actual_delivery_date AS DATE),
            CAST(expected_delivery_date AS DATE)
        )
    END AS delivery_delay_days,

    CASE
        WHEN actual_delivery_date IS NULL THEN NULL
        WHEN actual_delivery_date > expected_delivery_date THEN TRUE
        ELSE FALSE
    END AS is_late

FROM workspace.aegis_bronze.shipment;


-- ------------------------------------------------------------
-- SHIPMENT ITEM
--
-- Grain: one row per shipment item
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.shipment_item AS
SELECT
    CAST(shipment_item_id AS STRING) AS shipment_item_id,
    CAST(shipment_id AS STRING) AS shipment_id,
    CAST(order_item_id AS STRING) AS order_item_id,
    CAST(quantity AS BIGINT) AS quantity
FROM workspace.aegis_bronze.shipment_item;



-- ============================================================
-- 4. INVENTORY
-- ============================================================

-- ------------------------------------------------------------
-- INVENTORY SNAPSHOT
--
-- Grain:
--   one product × warehouse × snapshot date
--
-- available_quantity:
--   stock currently available after reservations.
--
-- stockout_flag:
--   deterministic indicator used by downstream inventory
--   health analytics.
-- ------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.aegis_silver.inventory AS
SELECT
    CAST(snapshot_date AS TIMESTAMP) AS snapshot_date,
    CAST(warehouse_id AS STRING) AS warehouse_id,
    CAST(product_id AS STRING) AS product_id,

    CAST(stock_quantity AS BIGINT) AS stock_quantity,
    CAST(reserved_quantity AS BIGINT) AS reserved_quantity,

    CAST(stock_quantity AS BIGINT)
        - CAST(reserved_quantity AS BIGINT)
        AS available_quantity,

    CASE
        WHEN (
            CAST(stock_quantity AS BIGINT)
            - CAST(reserved_quantity AS BIGINT)
        ) <= 0
        THEN TRUE
        ELSE FALSE
    END AS stockout_flag

FROM workspace.aegis_bronze.inventory_snapshot;



-- ============================================================
-- 5. RETURNS
-- ============================================================

CREATE OR REPLACE TABLE workspace.aegis_silver.return AS
SELECT
    CAST(return_id AS STRING) AS return_id,
    CAST(order_id AS STRING) AS order_id,
    CAST(product_id AS STRING) AS product_id,
    CAST(return_date AS TIMESTAMP) AS return_date,
    TRIM(return_reason) AS return_reason
FROM workspace.aegis_bronze.return;


-- ============================================================
-- END OF SILVER LAYER
-- ============================================================
--
-- Ground truth incidents intentionally excluded.
--
-- Next layer:
--
--   Silver
--      ↓
--   Gold business metrics
--
-- Planned Gold tables:
--   gold_supplier_performance
--   gold_delivery_performance
--   gold_inventory_health
--   gold_warehouse_performance
--   gold_carrier_performance
--   gold_supply_chain_risk
--
-- ============================================================